## Cleaning Summary

### What I changed

- I created a working copy of the `All Games` sheet and kept the raw source file unchanged.
- I dropped the fully empty column `Unnamed: 19`.
- I standardized all column names to lowercase with underscores and removed the `?` from `won`.
- I renamed several columns for clarity:
  - `no_m_score` → `no_mullen_score`
  - `no_m_lines` → `no_mullen_lines`
  - `19_trans` → `score_lvl_19_transition`
  - `19_l_start` → `lines_lvl_19_transition`
  - `29_trans` → `score_lvl_29_transition`
  - `29_l_start` → `lines_lvl_29_transition`
  - `39_trans` → `score_lvl_39_transition`
  - `39_l_start` → `lines_lvl_39_transition`
  - `level` → `start_level`
  - `cap` → `line_cap`
  - `sps` → `same_piece_sets_active`
- I standardized the inconsistent `round` label `L T8` to `LQ`.
- I reordered the columns into a cleaner analysis-ready structure.
- I saved the cleaned file separately in the `processed` folder.

### What I found

- I found **0 exact duplicate rows**, so no duplicate removal was needed.
- Most missing values in transition-related columns are expected because many players did not reach those stages:
  - `score_lvl_19_transition`
  - `lines_lvl_19_transition`
  - `score_lvl_29_transition`
  - `lines_lvl_29_transition`
  - `score_lvl_39_transition`
  - `lines_lvl_39_transition`
  - `post`
  - `post_post`
- `line_cap` is missing in many rows, but I kept it unchanged because this appears to be format- or rule-related and is not central to this project.
- `total_lines` has **18 real missing values**. Based on David’s clarification, these are mostly cases where the video feed disappeared, so only the final score was available.
- `topout_type` has **1 real missing value** (`game_id = 3635`). David confirmed this was a recording oversight.
- `no_mullen_score` and `no_mullen_lines` are intended to be used only for winners. The pattern mostly supports this rule, but I found **4 inconsistent cases** where these fields are filled although `won = No`.
- I checked the column dtypes and kept the current numeric types. Columns with missing numeric values remain floats, which is expected in pandas when `NaN` is present. I kept them this way because the dataset is already analysis-ready in its current form.
- I reviewed unusual zero-value cases in `final_score` and `total_lines`. Based on David’s clarification, these were treated as valid extreme cases rather than errors.

### What I did not change

- I did not fill missing values in `total_lines`.
- I did not fill the missing `topout_type` value.
- I did not change the 4 inconsistent `no_mullen` cases because I do not have enough evidence to correct them safely.
- I did not change unusual zero-value cases in `final_score` / `total_lines`, because David indicated these are likely true values rather than errors.
- I did not force category values like `Yes`, `No`, `Top 32`, `GFBR`, etc. into lowercase or underscore format, because they were already readable and consistent enough for analysis.
- I did not force additional type casting for numeric columns with missing values, because keeping them as floats is practical and preserves their usability for analysis.

### Working decision

The cleaned dataset is ready for analysis. I only applied changes that were clearly justified by empty columns, naming consistency, or confirmed label inconsistencies. I kept unresolved source-data issues unchanged and documented them instead of guessing.

In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

Qustions for David:
___
You mentioned that No M Score and No M Lines are supposed to appear only in games where players won. I found 4 cases where these fields are filled even though won = No. Could you clarify how these should be interpreted?

Cases:

match_id 87, game 2 — 2017 Semis
game_id 425: Alex Kerr, won = Yes, no_mullen_score = NaN, no_mullen_lines = NaN
game_id 426: Quaid, won = No, no_mullen_score = 643963, no_mullen_lines = 224
match_id 229, game 3 — 2020 WF
game_id 1275: CheeZ, won = Yes, no_mullen_score = NaN, no_mullen_lines = NaN
game_id 1276: Fractal, won = No, no_mullen_score = 989880, no_mullen_lines = 235
match_id 308, game 2 — 2021 LS
game_id 1903: Luis (POR), won = Yes, no_mullen_score = NaN, no_mullen_lines = NaN
game_id 1904: DanV, won = No, no_mullen_score = 965380, no_mullen_lines = 225
match_id 312, game 5 — 2021 WQ
game_id 1939: Somalian, won = No, no_mullen_score = 910980, no_mullen_lines = 230
game_id 1940: Sam Finch, won = Yes, no_mullen_score = NaN, no_mullen_lines = NaN
___


In [3]:
file_path = "../data/raw/public_ctwc_match_statistics.xlsx"
df_raw = pd.read_excel(file_path)
df = df_raw.copy()

In [4]:
excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['All Time Leaderboards',
 'Yearly Leaderboards',
 'Custom Leaderboard',
 'Player Statistics',
 'Player Profiles',
 'Player History',
 'Box Scores',
 'Head to Head',
 'MARFRAM STATS',
 'Year Stats',
 'All Games',
 'Validation',
 'Player Info',
 'Countries',
 'All Time Leaderboard Stats',
 'Yearly Leaderboard Stats',
 'Custom Leaderboard Stats',
 'Player Statistics Games']

In [5]:
df_raw = pd.read_excel(file_path, sheet_name="All Games")
df = df_raw.copy()

In [6]:
df.shape

(4202, 26)

In [7]:
df.columns.tolist()

['Player',
 'Game',
 'Playstyle',
 'Won?',
 'Final Score',
 'Total Lines',
 'No M Score',
 'No M Lines',
 '19 Trans',
 '19 L Start',
 '29 Trans',
 '29 L Start',
 '39 Trans',
 '39 L Start',
 'Topout Type',
 'Level',
 'Cap',
 'SPS',
 'Year',
 'Unnamed: 19',
 'Round',
 'Game Link',
 'Game ID',
 'Match ID',
 'Post',
 'Post Post']

In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [9]:
df.head()

,Player,Game,Playstyle,Won?,Final Score,Total Lines,No M Score,No M Lines,19 Trans,19 L Start,29 Trans,29 L Start,39 Trans,39 L Start,Topout Type,Level,Cap,SPS,Year,Unnamed: 19,Round,Game Link,Game ID,Match ID,Post,Post Post
0,Jonas,1,DAS,Yes,530034,195.0,339042.0,135.0,522034.0,190.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,2010,NaN,Finals,https://youtu.be/ZL4eRDOOP1I?t=24,1,1,NaN,NaN
1,Harry Hong,1,DAS,No,302118,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Natural,9,NaN,No,2010,NaN,Finals,https://youtu.be/ZL4eRDOOP1I?t=24,2,1,NaN,NaN
2,Jonas,2,DAS,Yes,544534,245.0,542531.0,243.0,443999.0,192.0,NaN,NaN,NaN,NaN,Intentional,9,NaN,No,2010,NaN,Finals,https://youtu.be/ZL4eRDOOP1I?t=595,3,1,NaN,NaN
3,Harry Hong,2,DAS,No,517590,227.0,NaN,NaN,483111.0,191.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,2010,NaN,Finals,https://youtu.be/ZL4eRDOOP1I?t=595,4,1,NaN,NaN
4,Jonas,1,DAS,Yes,164153,55.0,164153.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,Intentional,18,NaN,No,2011,NaN,Top 8,https://youtu.be/8sorxlk7QLs?t=14,5,2,NaN,NaN


In [10]:
df["Unnamed: 19"].head(10)

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: Unnamed: 19, dtype: float64

In [11]:
df["Unnamed: 19"].isna().sum()

np.int64(4202)

In [12]:
df["Unnamed: 19"].notna().sum()

np.int64(0)

In [13]:
df = df.drop(columns=["Unnamed: 19"])

In [14]:
df.columns.tolist()

['Player',
 'Game',
 'Playstyle',
 'Won?',
 'Final Score',
 'Total Lines',
 'No M Score',
 'No M Lines',
 '19 Trans',
 '19 L Start',
 '29 Trans',
 '29 L Start',
 '39 Trans',
 '39 L Start',
 'Topout Type',
 'Level',
 'Cap',
 'SPS',
 'Year',
 'Round',
 'Game Link',
 'Game ID',
 'Match ID',
 'Post',
 'Post Post']

In [15]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("?", "", regex=False)
    .str.replace(" ", "_", regex=False)
)

In [16]:
df.columns.tolist()

['player',
 'game',
 'playstyle',
 'won',
 'final_score',
 'total_lines',
 'no_m_score',
 'no_m_lines',
 '19_trans',
 '19_l_start',
 '29_trans',
 '29_l_start',
 '39_trans',
 '39_l_start',
 'topout_type',
 'level',
 'cap',
 'sps',
 'year',
 'round',
 'game_link',
 'game_id',
 'match_id',
 'post',
 'post_post']

In [17]:
df = df.rename(columns={
    "no_m_score": "no_mullen_score",
    "no_m_lines": "no_mullen_lines",
    "19_trans": "score_lvl_19_transition",
    "19_l_start": "lines_lvl_19_transition",
    "29_trans": "score_lvl_29_transition",
    "29_l_start": "lines_lvl_29_transition",
    "39_trans": "score_lvl_39_transition",
    "39_l_start": "lines_lvl_39_transition",
    "level": "start_level",
    "cap": "line_cap"
})

In [18]:
df.columns.tolist()

['player',
 'game',
 'playstyle',
 'won',
 'final_score',
 'total_lines',
 'no_mullen_score',
 'no_mullen_lines',
 'score_lvl_19_transition',
 'lines_lvl_19_transition',
 'score_lvl_29_transition',
 'lines_lvl_29_transition',
 'score_lvl_39_transition',
 'lines_lvl_39_transition',
 'topout_type',
 'start_level',
 'line_cap',
 'sps',
 'year',
 'round',
 'game_link',
 'game_id',
 'match_id',
 'post',
 'post_post']

In [19]:
df["sps"].value_counts(dropna=False)

sps
Yes    2142
No     2060
Name: count, dtype: int64

In [20]:
df[["sps", "year", "round", "player"]].head(20)

,sps,year,round,player
0,No,2010,Finals,Jonas
1,No,2010,Finals,Harry Hong
2,No,2010,Finals,Jonas
3,No,2010,Finals,Harry Hong
4,No,2011,Top 8,Jonas
5,No,2011,Top 8,Eli Markstrom
6,No,2011,Top 8,Jonas
7,No,2011,Top 8,Eli Markstrom
8,No,2011,Top 8,Robin Mihara
9,No,2011,Top 8,Ben Mullen


In [21]:
df = df.rename(columns={
    "sps": "same_piece_sets_active"
})

In [22]:
df.columns.tolist()

['player',
 'game',
 'playstyle',
 'won',
 'final_score',
 'total_lines',
 'no_mullen_score',
 'no_mullen_lines',
 'score_lvl_19_transition',
 'lines_lvl_19_transition',
 'score_lvl_29_transition',
 'lines_lvl_29_transition',
 'score_lvl_39_transition',
 'lines_lvl_39_transition',
 'topout_type',
 'start_level',
 'line_cap',
 'same_piece_sets_active',
 'year',
 'round',
 'game_link',
 'game_id',
 'match_id',
 'post',
 'post_post']

In [23]:
for col in ["round", "won", "playstyle", "topout_type", "line_cap", "same_piece_sets_active"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- round ---
round
Top 32    712
Top 64    626
WQ        514
Top 16    382
Top 8     366
L T8      256
WS        252
LQ        246
Semis     184
WF        130
GF        128
LS        126
LF        124
Finals    122
GFBR       34
Name: count, dtype: int64

--- won ---
won
Yes    2101
No     2101
Name: count, dtype: int64

--- playstyle ---
playstyle
Tap     1721
Roll    1316
DAS     1165
Name: count, dtype: int64

--- topout_type ---
topout_type
Natural        2726
Intentional    1455
Aggressive       20
NaN               1
Name: count, dtype: int64

--- line_cap ---
line_cap
NaN        3138
39 KSx2    1064
Name: count, dtype: int64

--- same_piece_sets_active ---
same_piece_sets_active
Yes    2142
No     2060
Name: count, dtype: int64


In [24]:
df[df["topout_type"].isna()][["game_id", "player", "year", "round", "game_link"]]

,game_id,player,year,round,game_link
3634,3635,Fractal,2024,Top 32,https://youtu.be/CkJ0UNI_7wg?t=1024


In [25]:
df[
    (df["won"] == "No") &
    (
        df["no_mullen_score"].notna() |
        df["no_mullen_lines"].notna()
    )
][["game_id", "player", "won", "no_mullen_score", "no_mullen_lines", "year", "round"]]

,game_id,player,won,no_mullen_score,no_mullen_lines,year,round
425,426,Quaid,No,643963.0,224.0,2017,Semis
1275,1276,Fractal,No,989880.0,235.0,2020,WF
1903,1904,DanV,No,965380.0,225.0,2021,LS
1938,1939,Somalian,No,910980.0,230.0,2021,WQ


In [26]:
df[df["match_id"].isin(
    df.loc[df["game_id"].isin([426, 1276, 1904, 1939]), "match_id"]
)][
    [
        "match_id",
        "game_id",
        "game",
        "player",
        "won",
        "final_score",
        "total_lines",
        "no_mullen_score",
        "no_mullen_lines",
        "year",
        "round",
        "game_link"
    ]
].sort_values(["match_id", "game", "game_id", "player"])

,match_id,game_id,game,player,won,final_score,total_lines,no_mullen_score,no_mullen_lines,year,round,game_link
422,87,423,1,Alex Kerr,No,500040,147.0,NaN,NaN,2017,Semis,https://youtu.be/vukZku_UVdg?t=383
423,87,424,1,Quaid,Yes,502095,183.0,502095.0,183.0,2017,Semis,https://youtu.be/vukZku_UVdg?t=383
424,87,425,2,Alex Kerr,Yes,677820,224.0,NaN,NaN,2017,Semis,https://youtu.be/vukZku_UVdg?t=901
425,87,426,2,Quaid,No,643963,224.0,643963.0,224.0,2017,Semis,https://youtu.be/vukZku_UVdg?t=901
426,87,427,3,Alex Kerr,Yes,681100,188.0,681100.0,188.0,2017,Semis,https://youtu.be/vukZku_UVdg?t=1531
427,87,428,3,Quaid,No,582621,192.0,NaN,NaN,2017,Semis,https://youtu.be/vukZku_UVdg?t=1531
1270,229,1271,1,CheeZ,Yes,674142,159.0,674142.0,159.0,2020,WF,https://youtu.be/CjfVFHy4Pr0?t=28
1271,229,1272,1,Fractal,No,611220,160.0,NaN,NaN,2020,WF,https://youtu.be/CjfVFHy4Pr0?t=28
1272,229,1273,2,CheeZ,No,35340,15.0,NaN,NaN,2020,WF,https://youtu.be/CjfVFHy4Pr0?t=482
1273,229,1274,2,Fractal,Yes,62320,28.0,62320.0,28.0,2020,WF,https://youtu.be/CjfVFHy4Pr0?t=482


In [27]:
pairs_to_check = [
    (87, 2),    # Quaid
    (229, 3),   # Fractal
    (308, 2),   # DanV
    (312, 5)    # Somalian
]

mask = pd.Series(False, index=df.index)

for match_id, game in pairs_to_check:
    mask |= ((df["match_id"] == match_id) & (df["game"] == game))

df.loc[mask, [
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "no_mullen_score",
    "no_mullen_lines",
    "year",
    "round"
]].sort_values(["match_id", "game_id"])

,match_id,game,game_id,player,won,final_score,total_lines,no_mullen_score,no_mullen_lines,year,round
424,87,2,425,Alex Kerr,Yes,677820,224.0,NaN,NaN,2017,Semis
425,87,2,426,Quaid,No,643963,224.0,643963.0,224.0,2017,Semis
1274,229,3,1275,CheeZ,Yes,997140,240.0,NaN,NaN,2020,WF
1275,229,3,1276,Fractal,No,989880,235.0,989880.0,235.0,2020,WF
1902,308,2,1903,Luis (POR),Yes,1038800,232.0,NaN,NaN,2021,LS
1903,308,2,1904,DanV,No,965380,225.0,965380.0,225.0,2021,LS
1938,312,5,1939,Somalian,No,910980,230.0,910980.0,230.0,2021,WQ
1939,312,5,1940,Sam Finch,Yes,958200,235.0,NaN,NaN,2021,WQ


In [28]:
df["round"] = df["round"].replace({"L T8": "LQ"})

In [29]:
df["round"].value_counts(dropna=False)

round
Top 32    712
Top 64    626
WQ        514
LQ        502
Top 16    382
Top 8     366
WS        252
Semis     184
WF        130
GF        128
LS        126
LF        124
Finals    122
GFBR       34
Name: count, dtype: int64

In [30]:
df.duplicated().sum()

np.int64(0)

In [31]:
df[df.duplicated(keep=False)].sort_values(["match_id", "game_id"]).head(20)

,player,game,playstyle,won,final_score,total_lines,no_mullen_score,no_mullen_lines,score_lvl_19_transition,lines_lvl_19_transition,score_lvl_29_transition,lines_lvl_29_transition,score_lvl_39_transition,lines_lvl_39_transition,topout_type,start_level,line_cap,same_piece_sets_active,year,round,game_link,game_id,match_id,post,post_post


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4202 entries, 0 to 4201
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   player                   4202 non-null   str    
 1   game                     4202 non-null   int64  
 2   playstyle                4202 non-null   str    
 3   won                      4202 non-null   str    
 4   final_score              4202 non-null   int64  
 5   total_lines              4184 non-null   float64
 6   no_mullen_score          2073 non-null   float64
 7   no_mullen_lines          2068 non-null   float64
 8   score_lvl_19_transition  3283 non-null   float64
 9   lines_lvl_19_transition  3281 non-null   float64
 10  score_lvl_29_transition  1245 non-null   float64
 11  lines_lvl_29_transition  1245 non-null   float64
 12  score_lvl_39_transition  32 non-null     float64
 13  lines_lvl_39_transition  32 non-null     float64
 14  topout_type              4201 non-n

In [33]:
df.isna().sum().sort_values(ascending=False)

score_lvl_39_transition    4170
lines_lvl_39_transition    4170
line_cap                   3138
post_post                  2963
score_lvl_29_transition    2957
lines_lvl_29_transition    2957
no_mullen_lines            2134
no_mullen_score            2129
post                        990
lines_lvl_19_transition     921
score_lvl_19_transition     919
game_link                   202
total_lines                  18
topout_type                   1
year                          0
match_id                      0
game_id                       0
round                         0
final_score                   0
same_piece_sets_active        0
playstyle                     0
start_level                   0
won                           0
game                          0
player                        0
dtype: int64

In [34]:
df.groupby("won")[["no_mullen_score", "no_mullen_lines"]].apply(lambda x: x.isna().sum())

,no_mullen_score,no_mullen_lines
won,,
No,2097,2097
Yes,32,37


In [35]:
df.groupby("won")[["no_mullen_score", "no_mullen_lines"]].apply(lambda x: x.notna().sum())

,no_mullen_score,no_mullen_lines
won,,
No,4,4
Yes,2069,2064


In [36]:
df[
    (df["won"] == "Yes") &
    (
        df["no_mullen_score"].isna() |
        df["no_mullen_lines"].isna()
    )
][[
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "no_mullen_score",
    "no_mullen_lines",
    "year",
    "round"
]].sort_values(["year", "match_id", "game"])

,match_id,game,game_id,player,won,final_score,total_lines,no_mullen_score,no_mullen_lines,year,round
114,24,1,115,Cameron Eure,Yes,642660,NaN,642660.0,NaN,2014,Top 16
277,57,2,278,TheMetalBeast,Yes,458626,NaN,458626.0,NaN,2016,Top 32
370,77,1,371,Trey Harrison,Yes,215621,72.0,NaN,NaN,2017,Top 32
391,80,3,392,Eli Markstrom,Yes,139383,56.0,NaN,NaN,2017,Top 16
404,83,3,405,Alex Kerr,Yes,778920,230.0,NaN,NaN,2017,Top 8
424,87,2,425,Alex Kerr,Yes,677820,224.0,NaN,NaN,2017,Semis
554,116,3,555,Harry Hong,Yes,598420,190.0,NaN,NaN,2018,Top 16
718,150,2,719,Chad,Yes,61560,23.0,NaN,NaN,2019,Top 32
1055,199,3,1056,Opaux,Yes,692960,186.0,NaN,NaN,2020,LQ
1274,229,3,1275,CheeZ,Yes,997140,240.0,NaN,NaN,2020,WF


In [37]:
df[df["total_lines"].isna()][[
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "year",
    "round"
]].sort_values(["year", "match_id", "game"])

,match_id,game,game_id,player,won,final_score,total_lines,year,round
114,24,1,115,Cameron Eure,Yes,642660,NaN,2014,Top 16
116,24,2,117,Cameron Eure,No,140600,NaN,2014,Top 16
118,24,3,119,Cameron Eure,No,468540,NaN,2014,Top 16
255,52,1,256,Adam Cornelius,No,51680,NaN,2016,Top 32
257,52,2,258,Adam Cornelius,No,298780,NaN,2016,Top 32
263,54,1,264,Vince,No,277630,NaN,2016,Top 32
265,54,2,266,Vince,No,82840,NaN,2016,Top 32
270,56,1,271,Chris Brady,No,428371,NaN,2016,Top 32
272,56,2,273,Chris Brady,No,35878,NaN,2016,Top 32
275,57,1,276,TheMetalBeast,No,265732,NaN,2016,Top 32


In [38]:
df[df["topout_type"].isna()][[
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "topout_type",
    "year",
    "round"
]]

,match_id,game,game_id,player,won,final_score,total_lines,topout_type,year,round
3634,531,3,3635,Fractal,Yes,1206060,278.0,NaN,2024,Top 32


In [39]:
df[
    (df["final_score"] == 0) | (df["total_lines"] == 0)
][[
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "year",
    "round"
]].sort_values(["year", "match_id", "game"])

,match_id,game,game_id,player,won,final_score,total_lines,year,round
361,75,1,362,Robin Mihara,No,2,0.0,2017,Top 32
1345,238,1,1346,B14NK,No,0,0.0,2020,WQ
2308,359,4,2309,Nevanator,No,0,0.0,2021,WQ
3049,454,1,3050,Tristop,No,0,0.0,2022,Top 16


In [40]:
zero_cases = df[
    (df["final_score"] == 0) | (df["total_lines"] == 0)
][[
    "match_id",
    "game",
    "game_id",
    "player",
    "won",
    "final_score",
    "total_lines",
    "year",
    "round",
    "game_link"
]].sort_values(["year", "match_id", "game"])

zero_cases

,match_id,game,game_id,player,won,final_score,total_lines,year,round,game_link
361,75,1,362,Robin Mihara,No,2,0.0,2017,Top 32,https://youtu.be/Sxz7rtCz48A?t=2000
1345,238,1,1346,B14NK,No,0,0.0,2020,WQ,https://youtu.be/jcpJOg5JlVg?t=41
2308,359,4,2309,Nevanator,No,0,0.0,2021,WQ,https://youtu.be/SsoH3vgyAow?t=9802
3049,454,1,3050,Tristop,No,0,0.0,2022,Top 16,https://youtu.be/nsxbSKTHQX8?t=1531


In [41]:
for col in ["won", "playstyle", "topout_type", "line_cap", "same_piece_sets_active"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- won ---
won
Yes    2101
No     2101
Name: count, dtype: int64

--- playstyle ---
playstyle
Tap     1721
Roll    1316
DAS     1165
Name: count, dtype: int64

--- topout_type ---
topout_type
Natural        2726
Intentional    1455
Aggressive       20
NaN               1
Name: count, dtype: int64

--- line_cap ---
line_cap
NaN        3138
39 KSx2    1064
Name: count, dtype: int64

--- same_piece_sets_active ---
same_piece_sets_active
Yes    2142
No     2060
Name: count, dtype: int64


In [42]:
df["round"].value_counts(dropna=False)

round
Top 32    712
Top 64    626
WQ        514
LQ        502
Top 16    382
Top 8     366
WS        252
Semis     184
WF        130
GF        128
LS        126
LF        124
Finals    122
GFBR       34
Name: count, dtype: int64

In [43]:
df = df[
    [
        "year",
        "round",
        "match_id",
        "game_id",
        "game",
        "player",
        "won",
        "playstyle",
        "final_score",
        "total_lines",
        "no_mullen_score",
        "no_mullen_lines",
        "score_lvl_19_transition",
        "lines_lvl_19_transition",
        "score_lvl_29_transition",
        "lines_lvl_29_transition",
        "score_lvl_39_transition",
        "lines_lvl_39_transition",
        "topout_type",
        "start_level",
        "line_cap",
        "same_piece_sets_active",
        "post",
        "post_post",
        "game_link"
    ]
]

In [44]:
df.columns.tolist()

['year',
 'round',
 'match_id',
 'game_id',
 'game',
 'player',
 'won',
 'playstyle',
 'final_score',
 'total_lines',
 'no_mullen_score',
 'no_mullen_lines',
 'score_lvl_19_transition',
 'lines_lvl_19_transition',
 'score_lvl_29_transition',
 'lines_lvl_29_transition',
 'score_lvl_39_transition',
 'lines_lvl_39_transition',
 'topout_type',
 'start_level',
 'line_cap',
 'same_piece_sets_active',
 'post',
 'post_post',
 'game_link']

In [45]:
df.head()

,year,round,match_id,game_id,game,player,won,playstyle,final_score,total_lines,no_mullen_score,no_mullen_lines,score_lvl_19_transition,lines_lvl_19_transition,score_lvl_29_transition,lines_lvl_29_transition,score_lvl_39_transition,lines_lvl_39_transition,topout_type,start_level,line_cap,same_piece_sets_active,post,post_post,game_link
0,2010,Finals,1,1,1,Jonas,Yes,DAS,530034,195.0,339042.0,135.0,522034.0,190.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24
1,2010,Finals,1,2,1,Harry Hong,No,DAS,302118,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24
2,2010,Finals,1,3,2,Jonas,Yes,DAS,544534,245.0,542531.0,243.0,443999.0,192.0,NaN,NaN,NaN,NaN,Intentional,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595
3,2010,Finals,1,4,2,Harry Hong,No,DAS,517590,227.0,NaN,NaN,483111.0,191.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595
4,2011,Top 8,2,5,1,Jonas,Yes,DAS,164153,55.0,164153.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,Intentional,18,NaN,No,NaN,NaN,https://youtu.be/8sorxlk7QLs?t=14


In [46]:
output_path = "../data/processed/ctwc_all_games_cleaned.xlsx"
df.to_excel(output_path, index=False)

output_path

'../data/processed/ctwc_all_games_cleaned.xlsx'

In [47]:
# watchlist:
watchlist = pd.DataFrame([
    {
        "match_id": 75,
        "game": 1,
        "game_id": 362,
        "player": "Robin Mihara",
        "column_to_check": "final_score / total_lines",
        "reason": "Unusual zero value; David said this is likely a true value, not an error",
        "game_link": df.loc[df["game_id"] == 362, "game_link"].iloc[0],
        "status": "to review"
    },
    {
        "match_id": 238,
        "game": 1,
        "game_id": 1346,
        "player": "B14NK",
        "column_to_check": "final_score / total_lines",
        "reason": "Unusual zero value; David said this is likely a true value, not an error",
        "game_link": df.loc[df["game_id"] == 1346, "game_link"].iloc[0],
        "status": "to review"
    },
    {
        "match_id": 359,
        "game": 4,
        "game_id": 2309,
        "player": "Nevanator",
        "column_to_check": "final_score / total_lines",
        "reason": "Unusual zero value; David said this is likely a true value, not an error",
        "game_link": df.loc[df["game_id"] == 2309, "game_link"].iloc[0],
        "status": "to review"
    },
    {
        "match_id": 454,
        "game": 1,
        "game_id": 3050,
        "player": "Tristop",
        "column_to_check": "final_score / total_lines",
        "reason": "Unusual zero value; David said this is likely a true value, not an error",
        "game_link": df.loc[df["game_id"] == 3050, "game_link"].iloc[0],
        "status": "to review"
    }
])